# Training Arabic Sentiment Model

Arabic customer reviews · **Positive / Negative / Neutral**.

## 1. Imports

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
sys.path.insert(0, str(ROOT.resolve()))

from src.Arabic_model import (
    ARABIC_FEATURE_NAMES,
    ARABIC_MODEL_NAMES,
    ArabicSentimentModel,
    build_arabic_baseline,
    build_arabic_pipeline,
    make_classifier,
    make_vectorizer,
    save_arabic_model,
)
from src.labels import ARABIC, NEGATIVE, NEUTRAL, POSITIVE, map_sentiment
from src.Preprocessing_pipeline import preprocess_arabic

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4f}".format)
LABELS = [NEGATIVE, NEUTRAL, POSITIVE]


## 2. Load Data

In [2]:
DATA = ROOT / "data" / "raw" / "arabic" / "Final_Data.csv"
df = pd.read_csv(DATA)
print(DATA.resolve())
df.head()


/home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/data/raw/arabic/Final_Data.csv


,review_description,rating,company
0,رائع,positive,talbat
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشكل اسرع,positive,talbat
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال بالشبكة..مع انه النت عندي تمام شو الحل??,negative,talbat
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,talbat
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكون فاتحه بقاله كده اكتر من شهر,negative,talbat


## 3. EDA

In [3]:
print("shape:", df.shape)
print("columns:", list(df.columns))
print(df["rating"].value_counts())
print(df.isnull().sum())

fig, ax = plt.subplots(figsize=(5, 3))
df["rating"].value_counts().reindex(["positive", "negative", "neutral"]).plot(
    kind="bar", ax=ax, color=["#2ca02c", "#d62728", "#7f7f7f"], rot=0
)
ax.set_title("Arabic class counts")
plt.tight_layout()
plt.show()


shape: (40046, 3)
columns: ['review_description', 'rating', 'company']
rating
positive    23921
negative    14200
neutral      1925
Name: count, dtype: int64
review_description    1
rating                0
company               0
dtype: int64


/tmp/ipykernel_56862/2391177505.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Cleaning

Drop null text, conflicting labels, then duplicate reviews. Neutral stays — it is in the data.

In [4]:
df = df.dropna(subset=["review_description"]).copy()
df["label"] = df["rating"].map(lambda x: map_sentiment(x, ARABIC))
conflict = df.groupby("review_description")["label"].nunique().loc[lambda s: s > 1].index
print("conflicting texts:", len(conflict))
df = df[~df["review_description"].isin(conflict)]
df = df.drop_duplicates(subset=["review_description"])
print(df["label"].value_counts())


conflicting texts: 95
label
Positive    23119
Negative    13900
Neutral      1889
Name: count, dtype: int64


## 5. Preprocessing

Arabic only: URLs → punctuation → tashkeel → tatweel → Alef → Ya → whitespace.
No English tokenizer or English stopwords. `مش` is kept (`مش حلو` must not become `حلو`).

In [5]:
df["clean_text"] = df["review_description"].map(preprocess_arabic)
df = df[df["clean_text"].str.len() > 0]
print("مش kept:", "مش" in preprocess_arabic("الخدمة مش حلوة").split())
df[["review_description", "clean_text", "label"]].head(3)


مش kept: True


,review_description,clean_text,label
0,رائع,رائع,Positive
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشكل اسرع,برنامج رائع جدا يساعد علي تلبيه الاحتياجات بشكل اسرع,Positive
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال بالشبكة..مع انه النت عندي تمام شو الحل??,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال بالشبكة مع انه النت عندي تمام شو الحل,Negative


## 6. Train/Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
print("train", len(X_train), "val", len(X_val), "test", len(X_test))


train 31108 val 6222 test 7778


## 7. Feature Extraction

Word TF-IDF, word n-grams (1–2), and character n-grams `TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))` — useful for Arabic spelling variation.

## 8. Baseline Model

Word TF-IDF + Logistic Regression. Metrics are **macro**-averaged because Neutral is rare.

In [7]:
def scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

baseline = build_arabic_baseline()
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
print(classification_report(y_test, base_pred, labels=LABELS, digits=4, zero_division=0))
print(pd.Series(scores(y_test, base_pred), name="baseline"))


              precision    recall  f1-score   support

    Negative     0.8300    0.7837    0.8062      2779
     Neutral     0.3636    0.0212    0.0400       378
    Positive     0.8365    0.9290    0.8803      4621

    accuracy                         0.8330      7778
   macro avg     0.6767    0.5780    0.5755      7778
weighted avg     0.8112    0.8330    0.8130      7778

Accuracy    0.8330
Precision   0.6767
Recall      0.5780
F1          0.5755
Name: baseline, dtype: float64


## 9. Model Experiments

In [8]:
rows = []
for feature_name in ARABIC_FEATURE_NAMES:
    vec = make_vectorizer(feature_name)
    Xtr = vec.fit_transform(X_fit)
    Xva = vec.transform(X_val)
    for model_name in ARABIC_MODEL_NAMES:
        clf = make_classifier(model_name)
        clf.fit(Xtr, y_fit)
        pred = clf.predict(Xva)
        row = {"Model": model_name, "Features": feature_name, **scores(y_val, pred)}
        rows.append(row)
        print(f"val {model_name:20} {feature_name:20} macro-F1={row['F1']:.4f}")

val_results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
val_results


val Naive Bayes          Word TF-IDF          macro-F1=0.5547


val Logistic Regression  Word TF-IDF          macro-F1=0.5629
val Linear SVM           Word TF-IDF          macro-F1=0.5795


val Naive Bayes          Word n-grams         macro-F1=0.5623


val Logistic Regression  Word n-grams         macro-F1=0.5642


val Linear SVM           Word n-grams         macro-F1=0.5833


val Naive Bayes          Character n-grams    macro-F1=0.5569


val Logistic Regression  Character n-grams    macro-F1=0.5972


val Linear SVM           Character n-grams    macro-F1=0.6048


,Model,Features,Accuracy,Precision,Recall,F1
0,Linear SVM,Character n-grams,0.8436,0.6609,0.6028,0.6048
1,Logistic Regression,Character n-grams,0.8494,0.7106,0.5984,0.5972
2,Linear SVM,Word n-grams,0.8282,0.6068,0.5839,0.5833
3,Linear SVM,Word TF-IDF,0.8256,0.6204,0.5806,0.5795
4,Logistic Regression,Word n-grams,0.8320,0.5827,0.5740,0.5642
5,Logistic Regression,Word TF-IDF,0.8301,0.5756,0.5722,0.5629
6,Naive Bayes,Word n-grams,0.8333,0.5532,0.5729,0.5623
7,Naive Bayes,Character n-grams,0.8224,0.5412,0.5748,0.5569
8,Naive Bayes,Word TF-IDF,0.8235,0.5472,0.5643,0.5547


## 10. Evaluation

In [9]:
best_row = val_results.iloc[0]
best_model_name, best_feature_name = best_row["Model"], best_row["Features"]
print("validation winner:", best_model_name, "+", best_feature_name)

best = build_arabic_pipeline(best_feature_name, best_model_name)
best.fit(X_train, y_train)
y_pred = best.predict(X_test)
print("\nHeld-out TEST")
print(classification_report(y_test, y_pred, labels=LABELS, digits=4, zero_division=0))
print(pd.Series(scores(y_test, y_pred), name="test"))

cm = confusion_matrix(y_test, y_pred, labels=LABELS)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Arabic — test confusion matrix")
plt.tight_layout()
plt.show()


validation winner: Linear SVM + Character n-grams



Held-out TEST
              precision    recall  f1-score   support

    Negative     0.8107    0.8309    0.8207      2779
     Neutral     0.2121    0.0370    0.0631       378
    Positive     0.8608    0.9061    0.8829      4621

    accuracy                         0.8370      7778
   macro avg     0.6279    0.5913    0.5889      7778
weighted avg     0.8114    0.8370    0.8208      7778

Accuracy    0.8370
Precision   0.6279
Recall      0.5913
F1          0.5889
Name: test, dtype: float64


/tmp/ipykernel_56862/1180728414.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Error analysis

| Example | Likely gold | Typical failure |
|---|---|---|
| الفيلم مش وحش | Positive / Neutral | Negation `مش` + `وحش` |
| Very short praise (`رائع`) | Positive | Usually OK |
| Dialect / slang | varies | Sparse in TF-IDF |

In [10]:
tmp = ArabicSentimentModel(best)
for text in ("الفيلم مش وحش", "مش حلو", "الخدمة ممتازة والتجربة كانت رائعة"):
    print(f"{tmp.predict(text):10}  {text}")


Negative    الفيلم مش وحش
Negative    مش حلو
Positive    الخدمة ممتازة والتجربة كانت رائعة


## 11. Best Model

In [11]:
print(best)
print(best_row.to_dict())


Pipeline(steps=[('vec',
                 TfidfVectorizer(analyzer='char_wb', max_features=30000,
                                 ngram_range=(3, 5))),
                ('clf', LinearSVC())])
{'Model': 'Linear SVM', 'Features': 'Character n-grams', 'Accuracy': 0.8436194149791064, 'Precision': 0.6608755427561852, 'Recall': 0.6027698497956665, 'F1': 0.6048185301205945}


## 12. Save Model

In [12]:
path = save_arabic_model(best, ROOT / "models" / "Arabic_model_weights.pkl")
print("saved", path.resolve())


saved /home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/models/Arabic_model_weights.pkl


## 13. Example Predictions

In [13]:
model = ArabicSentimentModel.load(path)
for text in (
    "الفيلم رائع جدا",
    "الخدمة كانت سيئة",
    "الخدمة ممتازة والتجربة كانت رائعة",
    "الخدمة سيئة جدا ولن أكرر التجربة",
):
    print(f"{model.predict(text):10}  {text}")


Positive    الفيلم رائع جدا
Negative    الخدمة كانت سيئة
Positive    الخدمة ممتازة والتجربة كانت رائعة
Negative    الخدمة سيئة جدا ولن أكرر التجربة
